### **Connect Google Drive**

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### **Import Libraries & Install Dependencies**

In [2]:
import re
import pandas as pd
import numpy as np
from tqdm import tqdm

In [3]:
!pip install scikit-multilearn -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.4/89.4 kB 2.9 MB/s eta 0:00:00


### **Import Dataset**

In [4]:
#Read Dataset
df = pd.read_csv('/content/drive/MyDrive/Colab Datasets/in_hf.csv')
print(df.head())

                                                text  labels     source  \
0  @USER.wood17 knp lo gak berani bersumpah dan b...       1  Instagram   
1  haha, somad somad. Muka dekil otak 0% , kok ya...       1  Instagram   
2  hahaha, kaum sableng 212 kl berita begini mrk ...       1  Instagram   
3  hahaha, makin stress aja  ni umat sableng, dlu...       1  Instagram   
4       HIDUP PSI = partai SAMPAH indonesia..... ...       1  Instagram   

        dataset  nb_annotators  
0  ID_instagram              3  
1  ID_instagram              3  
2  ID_instagram              3  
3  ID_instagram              3  
4  ID_instagram              3  


In [5]:
print(df.columns.tolist())

['text', 'labels', 'source', 'dataset', 'nb_annotators']


### **Word List**

In [6]:
SLANG_WORDS = {
    # -- A1. Kata ganti (pronoun) informal --------------------------------
    "gue", "gw", "gua",
    "lo", "lu", "elo", "elu", "loe", "lw",
    "doi", "die", "dye",
    "kite",                     # kita (Betawi)
    "ane", "ente",              # saya / kamu (prokem Arab-Betawi)
    "akika",                    # aku (bahasa banci/gay slang)
    "aq",                       # aku (alay) [R3 Salsabila]
    "qt",                       # kita (alay) [R3]
    "dy",                       # dia (alay) [R3]

    # -- A2. Tawa & ekspresi imitatif [R1 Budiasa - Imitative] -----------
    "wkwk", "wkwkwk", "wkwkwkwk", "wkwkwkwkwk",
    "wkakak", "kwkwk",
    "xixi", "xixixi",
    "hehe", "hihi", "huhu", "hoho",
    "hahahah",
    "hiks", "hikz",
    "zzz",                      # mengantuk/bosan
    "uwu",                      # ekspresi gemas (Gen-Z)
    "owo",                      # ekspresi terkejut/gemas (Gen-Z)

    # -- A3. Sapaan & panggilan [R1][R3] ----------------------------------
    "cuy", "coy", "cok",
    "gaes",                     # guys (serapan)
    "lur",                      # saudara/teman (gaul)
    "sob",                      # sahabat
    "beb", "beby",              # baby (panggilan)
    "say",                      # sayang (singkatan panggilan)
    "bestie",                   # best friend
    "bff",                      # best friend forever
    "anjay", "anjir", "njir", "njay",
    "astaga", "astg",
    "bokap", "nyokap", "bonyok",
    "cewek", "cewe", "cowok", "cowo",
    "bang", "abang",
    "kak", "kaka", "ka", "kk",
    "brur",                     # brother (alay)
    "ngab",                     # abang (walikan)

    # -- A4. Fresh & Creative (kata baru kreatif) [R1] -------------------
    "baper",     # bawa perasaan
    "mager",     # malas gerak
    "gabut",     # ga ada buat (bosan)
    "bucin",     # budak cinta
    "julid",     # nyinyir/iri
    "kepo",      # knowing every particular object -> penasaran
    "santuy",    # santai (walikan)
    "kuy",       # yuk (walikan)
    "sabi",      # bisa (walikan)
    "kece",      # keren (gaul)
    "gokil",     # gila/keren
    "alay",      # norak/lebay
    "lebay",     # berlebihan
    "bete",      # kesal (BT)
    "galau",     # bimbang/sedih
    "mantul",    # mantap betul
    "mantap",    # bagus sekali (gaul)
    "asoy",      # asyik (Sunda gaul)
    "slebew",    # ekspresi kagum Gen-Z 2023+
    "gaskeun",   # ayo lakukan (gas + keun Sunda)
    "gaspol",    # lakukan sepenuhnya
    "gemoy",     # menggemaskan (Gen-Z 2023-2024)
    "gemes",     # gemas
    "cemungud",  # semangat (alay)
    "sultan",    # orang kaya (gaul)
    "cuan",      # keuntungan/uang (gaul)
    "gabisa",    # tidak bisa (klipping)
    "gamau",     # tidak mau (klipping)
    "gausah",    # tidak usah (klipping)
    "yaman",     # lumayan (walikan)
    "jamet",     # jawa metal -> uncool/kampungan
    "jomblo", "jomlo",
    "gebetan",   # orang yang ditaksir
    "naksir",    # suka seseorang
    "modus",     # motif tersembunyi / PDKT
    "healing",   # pulih mental (serapan jadi gaul)
    "bobok",     # tidur (bayi -> gaul)
    "caper",     # cari perhatian
    "minder",    # rendah diri
    "pamer",     # memamerkan
    "emosi",     # marah (gaul)
    "oce",       # oke (gaul)
    "receh",     # remeh/murahan
    "gercep",    # gerak cepat
    "pansos",    # panjat sosial
    "ngaret",    # terlambat/molor

    # -- A5. Flippant (santai/meremehkan) [R1] ----------------------------
    "garing",    # tidak lucu
    "jayus",     # humor gagal
    "cuek",      # acuh tak acuh
    "cupu",      # culun
    "culun",     # kampungan
    "dodol",     # bodoh (gaul)
    "telmi",     # telat mikir
    "kuper",     # kurang pergaulan
    "narsis",
    "centil",    # genit
    "jijay",     # jijik (pengucapan gaul)
    "jijik",
    "bawel",     # cerewet
    "ngoceh",    # bicara terus
    "nyinyir",   # berkomentar negatif
    "nyebelin",  # menyebalkan
    "ngeselin",  # menyebalkan
    "gaje",      # ga jelas
    "asbun",     # asal bunyi
    "ngawur",    # sembarangan
    "ngasal",    # asal-asalan
    "norak",     # kampungan/kuno
    "cakep",     # tampan/cantik (gaul)
    "imut",      # lucu/menggemaskan (gaul)
    "capek", "cape",
    "laper",
    "ngantuk",
    "mumet",     # pusing
    "bosen",     # bosan
    "kangen",    # rindu
    "awkward",   # canggung (serapan jadi gaul)

    # -- A6. Kata kerja informal [R3][R4] ---------------------------------
    "ngerti", "ngegas", "ngeles",
    "ngampus", "ngekos", "nongkrong",
    "woles", "selow",
    "mpe", "ampe", "sampe",
    "emang", "gimana", "gitu", "gini",
    "kalo", "banget",
    "abis", "dapet", "ngomong",
    "ngomongin", "nyindir", "ngejek",
    "ngehina", "nyalahin",
    "nganggep",
    "ngakak", "ngikik",
    "ngedown",    # merasa sedih (gaul)
    "ngeprank",   # melakukan prank
    "ngerekam",   # merekam (informal)
    # nge- prefix digital (non-hyphenated forms, paling umum di sosmed)
    "ngeblock",   # memblokir
    "ngestalk",   # menguntit akun
    "ngehack",    # meretas
    "ngeexpose", "ngexpose",
    "ngedm",      # DM seseorang
    "ngeshare",   # berbagi
    "ngelike",    # menyukai
    "ngerepost",  # repost
    "ngecancel",  # cancel culture

    # -- A7. Partikel & filler gaul [R3] ----------------------------------
    "nah", "lah", "dong", "sih", "nih", "tuh", "deh", "kok", "kan",
    "yaudah", "yowes", "yowis",
    "kek",        # kayak/seperti (filler)
    "makasih", "makasi",
    "sip", "iyap", "iyaps",
    "yoi", "yoii",

    # -- A8. Walikan (pembalikan silabis) [R1 Budiasa, Arifin 2022] ------
    "tubir",      # ribut
    "nakam",      # makan
    "omed",       # demo
    "toper",      # protes
    "kamer",      # mereka (walikan Jawa)

    # -- A9. Akronim gaul Indonesia [R1 Budiasa - Acronym] ---------------
    "otw",        # on the way (sangat umum di Indonesia)
    "gpp", "gppa",    # gak apa-apa
    "ttm",        # teman tapi mesra
    "cmiiw",      # correct me if I'm wrong
    "egp",        # emang gue pikirin
    "php",        # pemberi harapan palsu
    "jones",      # jomblo ngenes
    "clbk",       # cinta lama bersemi kembali
    "curcol",     # curahan hati colongan
    "folbek",     # follow back
    "kpop", "kpopers",
    "drakor",     # drama Korea
    "fyp",        # for your page (TikTok Indonesia)
    "pdkt",       # pendekatan (akronim gaul Indonesia)
    "mimin",      # admin/moderator (gaul sosmed)

    # -- A10. Kata serapan sosmed yang jadi gaul --------------------------
    "netizen", "warganet",
    "ngehype", "ngetrend", "ngeviral",

        # -- B1. Emosi & mental state (spesifik sosmed/informal) --------------
    "toxic",
    "vibes",
    "overthinking",
    "triggered",
    "insecure",
    "burnout",
    "gaslight", "gaslighting",
    "manipulative",
    "narcissist",
    "redflag",           # single-token form (frasa -> SLANG_PHRASES)
    "overwhelmed",
    "drained",
    "numb",
    "spiraling",
    "doomscrolling",
    "hyperfixation",
    "ick",               # ekspresi jijik tiba-tiba terhadap seseorang

    # -- B2. Penilaian & sikap (Gen-Z/sosmed) ----------------------------
    "cringe", "cringey",
    "valid",
    "slay",
    "iconic",
    "aesthetic",
    "wholesome",
    "petty",
    "shady",
    "salty",
    "extra",
    "basic",
    "lowkey", "highkey",
    "mid",
    "based",
    "ratio", "ratioed",
    "sus",
    "savage",
    "badass",
    "loser",
    "wannabe",
    "fake",
    "overrated", "underrated", "overhyped", "washed",
    "delusional", "delulu",
    "unhinged", "chaotic",
    "obsessed", "clingy", "possessive",
    "cooked",            # dalam masalah besar
    "flopped",           # gagal total
    "snatched",          # tampil luar biasa
    "npc",               # non-playable character -> pasif
    "clowned",           # bertindak bodoh
    "goated",            # GOAT-level
    "sigma",             # sigma male (internet culture)
    "copium",            # cope + opium
    "mogged",            # outperformed

    # -- B3. Hubungan & sosial (spesifik Gen-Z/sosmed) -------------------
    "ghosting", "ghosted",
    "crush",
    "squad",
    "fomo",              # fear of missing out
    "yolo",              # you only live once
    "hangout",
    "dating", "flirting",
    "friendzone", "friendzoned",
    "situationship",
    "rizz", "rizzing", "rizzy", "rizzler",
    "simp", "simping",
    "pickme",
    "bodycount",
    "girlboss",
    "boysober",
    "benching",
    "breadcrumbing",
    "lovebombing",
    "glowup",

    # -- B4. Aktivitas digital (spesifik informal/negatif) ----------------
    "hype",
    "catfish", "catfished",
    "troll", "trolling",
    "doxing", "doxxing",
    "spam", "spamming",
    "bot",
    "astroturfing",
    "shadowbanned",
    "clout",
    "flexing", "flex",
    "cancel", "cancelled", "canceling", "canceled",
    "exposed", "exposing",

    # -- B5. Gen-Z internet culture (token, bukan singkatan) --------------
    "nocap", "cap", "capping",
    "periodt",
    "tea",               # gossip
    "spill",             # spill the tea = cerita gosip
    "shade",             # sindir halus
    "pookie",            # term of endearment
    "era",               # "in my __ era" (Gen-Z)
    "bet",               # affirmation ("bet!" = oke!)
    "bussin",            # sangat enak/bagus
    "sheesh",
    "gatekeep", "gatekeeping",
    "brat",              # tren slang 2024
    "demure",            # viral 2024
    "girlypop",
}

In [7]:
SLANG_PHRASES = {

    # -- Frasa gaul Indonesia asli ----------------------------------------
    "ga ada otak", "gak ada otak",
    "dasar lo",
    "emang lo siapa",
    "mau apa lo",
    "gabut banget",
    "gak jelas",
    "ga jelas",
    "pdkt sama",

    # -- Frasa code-mixed (Inggris dalam konteks Indonesia) ---------------
    "no cap",
    "red flag", "green flag", "beige flag",
    "main character", "main character energy",
    "understood the assignment",
    "touch grass",
    "skill issue",
    "rent free", "living rent free",
    "girl math", "girl dinner",
    "chronically online",
    "pick me", "pick me girl",
    "friend zone",
    "love bombing",
    "talking stage",
    "soft launch", "hard launch",
    "say less",
    "on god",
    "very demure", "very mindful",
    "roman era",
    "brat summer",
    "caught in 4k",
    "ate and left no crumbs", "left no crumbs",
    "its giving", "it's giving",
    "plot twist",
    "character development",
    "girl boss",
    "vibe check",
    "gassed up",
    "body count",
    "glow up",
    "not me",
    "for real for real",
    "hits different",
    "gaslight girlboss gatekeep",
    "this ain't it",
    "and i oop",
}


In [8]:
ABBREV_WORDS = {
    # ====================================================================
    # POLA 1: PENGHILANGAN VOKAL (konsonan-dominan)
    # ====================================================================

    # -- Kata ganti & kata tanya ------------------------------------------
    "yg",                           # yang
    "sy",                           # saya
    "km",                           # kamu
    "mrk",                          # mereka
    "kpn",                          # kapan
    "dmn", "dmana",                 # di mana
    "gmn",                          # bagaimana
    "knp",                          # kenapa
    "spy",                          # supaya
    "sp",                           # siapa
    "skrg", "skrang",               # sekarang
    "stlh",                         # setelah
    "sblm",                         # sebelum
    "sbnrnya", "sbnarnya",          # sebenarnya
    "sebenrnya", "sbenrnya",
    "mksdnya",                      # maksudnya
    "kyknya",                       # kayaknya
    "pknnya",                       # pokoknya
    "sbg",                          # sebagai
    "smua",                         # semua
    "stiap",                        # setiap
    "mgkin",                        # mungkin
    "kmn",                          # kemana
    "drpd",                         # daripada
    "ttpi",                         # tetapi
    "tnpa",                         # tanpa
    "smpe",                         # sampai
    "pdhl",                         # padahal
    "wlpn",                         # walaupun
    "mnrt",                         # menurut
    "bgmn",                         # bagaimana
    "bgitu",                        # begitu
    "sklpn",                        # sekalipun

    # -- Negasi -----------------------------------------------------------
    "gak", "ga", "g", "gk",
    "ngga", "nggak", "ndak", "kagak", "enggak",
    "gapapa", "gpp",
    "gamau", "gamu",
    "gabisa", "gabsa",
    "gausah",
    "gabtau", "gatau", "gtw",
    "gasuka", "gasih",
    "gaada",                        # tidak ada
    "ganiat",                       # tidak niat
    "bkn",                          # bukan
    "blm",                          # belum
    "jgn",                          # jangan
    "krg",                          # kurang
    "lbh",                          # lebih
    "bnyk",                         # banyak
    "sdkt",                         # sedikit

    # -- Kata kerja & status ----------------------------------------------
    "udh", "udah", "sdh", "dah",    # sudah
    "msh",                          # masih
    "mo",                           # mau
    "bs",                           # bisa
    "hrs",                          # harus
    "brg",                          # bareng
    "pg",                           # pergi
    "dtg",                          # datang
    "plng",                         # pulang
    "tdr",                          # tidur
    "mkn",                          # makan
    "mnm",                          # minum
    "blj",                          # belajar
    "krj",                          # kerja
    "ntn",                          # nonton
    "klr",                          # keluar
    "msk",                          # masuk
    "jwb",                          # jawab
    "tny",                          # tanya
    "tlg",                          # tolong
    "bwt",                          # buat
    "pke", "pk",                    # pakai
    "pny",                          # punya
    "byar", "byr",                  # bayar
    "kluar",                        # keluar (varian)
    "dpt",                          # dapat
    "brkt",                         # berangkat
    "ktmu",                         # ketemu
    "nyoba",                        # mencoba (informal)
    "ngerjain",                     # mengerjakan (informal)
    "balikin",                      # mengembalikan (informal)
    "diceritain",                   # diceritakan (informal)
    "nemenin",                      # menemani (informal)

    # -- Partikel & konjungsi disingkat -----------------------------------
    "jg",                           # juga
    "aj", "aja",                    # saja
    "sm",                           # sama
    "tp", "tpi",                    # tapi
    "krn", "karna", "krna",         # karena
    "klo", "klu",                   # kalau
    "sprt", "spt",                  # seperti
    "ato",                          # atau
    "kyk",                          # kayak
    "trs", "trus",                  # terus
    "jd", "jdi",                    # jadi
    "pd",                           # pada
    "dr", "dri",                    # dari
    "dgn", "dg", "dngn",            # dengan
    "dlm",                          # dalam
    "utk",                          # untuk
    "ttg",                          # tentang
    "thd",                          # terhadap
    "ttap",                         # tetap
    "brrti",                        # berarti
    "bkl",                          # bakal

    # -- Preposisi singkat (hanya cocok jika isolated token) --------------
    "d",                            # di
    "k",                            # ke

    # -- Intensifier disingkat --------------------------------------------
    "bgt", "bngt", "bngd",          # banget
    "sgt",                          # sangat
    "plg",                          # paling
    "bnr", "bner",                  # benar

    # -- Waktu & tempat ---------------------------------------------------
    "lg", "lgi",                    # lagi
    "td",                           # tadi
    "dlu",                          # dulu
    "dpn",                          # depan
    "blkg",                         # belakang
    "bwh",                          # bawah
    "ats",                          # atas
    "sna", "sni",                   # sana / sini
    "sblah",                        # sebelah
    "mlm",                          # malam
    "kmren", "kmarin",              # kemarin
    "bsok", "bsk",                  # besok
    "mgu", "mgg",                   # minggu
    "bln",                          # bulan
    "thn",                          # tahun
    "jm",                           # jam
    "malem",                        # malam (informal)

    # -- Kata benda umum --------------------------------------------------
    "org",                          # orang
    "tmn", "tmen",                  # teman
    "klg",                          # keluarga
    "hp",                           # handphone
    "nmr",                          # nomor
    "jln", "jl",                    # jalan
    "tggl", "tgl",                  # tanggal
    "kmps",                         # kampus
    "mhsw",                         # mahasiswa
    "dsn",                          # dosen
    "rmh",                          # rumah
    "kmpung",                       # kampung
    "mbl",                          # mobil
    "mtr",                          # motor
    "pkt",                          # paket
    "ktr",                          # kantor

    # -- Kota (singkatan informal) ----------------------------------------
    "jkt",                          # jakarta
    "sby",                          # surabaya
    "bdg",                          # bandung
    "mdn",                          # medan
    "yk", "jogja",                  # yogyakarta
    "bks",                          # bekasi
    "dpk",                          # depok
    "tgr",                          # tangerang
    "smr",                          # semarang
    "mksr",                         # makassar
    "plbg",                         # palembang

    # -- Platform & aplikasi ----------------------------------------------
    "wa",                           # whatsapp
    "ig", "insta",                  # instagram
    "tt",                           # tiktok
    "yt", "ytb",                    # youtube
    "fb",                           # facebook
    "tele", "tg",                   # telegram
    "twt",                          # twitter/x
    "sptfy",                        # spotify
    "dm", "pm", "dms",              # direct/private message

    # -- Singkatan formal yang dipakai informal ---------------------------
    "dll", "dkk", "dst", "dsb", "tsb",

    # -- Respons cepat (Indonesia) ----------------------------------------
    "ok", "oke", "oce",
    "mksh", "mks",                  # makasih
    "mf", "maap",                   # maaf
    "sori", "sry",
    "pls", "plz",
    "hbs",                          # habis
    "jls",                          # jelas
    "smngt",                        # semangat
    "otw",                          # on the way
    "pdkt",                         # pendekatan

    # -- Singkatan Inggris umum dalam code-mixed Indonesia ----------------
    "thx", "tks", "tq", "ty", "tnx",
    "lmk",                          # let me know
    "omw",                          # on my way
    "wdym",                         # what do you mean
    "wbu",                          # what about you
    "hbu",                          # how about you
    "jk", "jkjk",                   # just kidding
    "nvm",                          # nevermind
    "yw",                           # you're welcome
    "hmu",                          # hit me up
    "ttyl",                         # talk to you later
    "gtg", "g2g",                   # got to go
    "bbl",                          # be back later
    "brb",                          # be right back
    "omg",                          # oh my god
    "wtf",                          # what the f
    "wth",                          # what the heck
    "rofl",                         # rolling on floor laughing
    "lol",                          # laugh out loud
    "lmao", "lmfao",
    "omfg",
    "lmaooo", "lolll",              # elongated forms
    "asap",                         # as soon as possible
    "pov",                          # point of view
    "tw", "cw",                     # trigger/content warning
    "iykyk",                        # if you know you know
    "tbt",                          # throwback thursday
    "ootd",                         # outfit of the day
    "grwm",                         # get ready with me
    "fr", "frfr",
    "ong",                          # on god
    "istg",                         # i swear to god
    "idc", "idk",
    "smh",                          # shaking my head
    "ngl",                          # not gonna lie
    "np",                           # no problem
    "btw", "fyi",
    "imo", "imho", "ikr",
    "tbh",                          # to be honest
    "ily", "ilysm",
    "tysm",
    "stfu",                         # toxic abbreviation -- hate speech marker
    "kys", "kms",                   # toxic abbreviations -- hate speech markers
    "idgaf",
    "wyd", "wya",
    "sup",
    "icymi",                        # in case you missed it
    "js",                           # just saying
    "gg",                           # good game
    "afk",                          # away from keyboard
    "irl",                          # in real life
    "nsfw", "sfw",

    # -- Reduplikasi alay (juga dideteksi via regex) ----------------------
    "jalan2",                       # jalan-jalan
    "makan2",                       # makan-makan
    "temen2",                       # teman-teman
    "main2",                        # main-main
    "santai2",
    "jln2",
    "malem2",                       # malam-malam
    "pagi2",                        # pagi-pagi
    "diem2",                        # diam-diam
    "lama2",                        # lama-lama
}

In [9]:
overlap = SLANG_WORDS & ABBREV_WORDS
if overlap:
    print(f"[INFO] Overlap SLANG_WORDS ∩ ABBREV_WORDS ({len(overlap)} kata): {sorted(overlap)}")
    print("       Overlap disengaja: toxic markers / akronim yang juga berfungsi sebagai slang.")
print(f"[INFO] SLANG_WORDS  : {len(SLANG_WORDS)} kata")
print(f"[INFO] SLANG_PHRASES: {len(SLANG_PHRASES)} frasa")
print(f"[INFO] ABBREV_WORDS : {len(ABBREV_WORDS)} kata")

[INFO] Overlap SLANG_WORDS ∩ ABBREV_WORDS (7 kata): ['gabisa', 'gamau', 'gausah', 'gpp', 'oce', 'otw', 'pdkt']
       Overlap disengaja: toxic markers / akronim yang juga berfungsi sebagai slang.
[INFO] SLANG_WORDS  : 362 kata
[INFO] SLANG_PHRASES: 54 frasa
[INFO] ABBREV_WORDS : 314 kata


### **Minimal Clean**

In [10]:
def minimal_clean(text):
    if pd.isna(text):
        return ""
    t = str(text).strip()
    # Curly quotes & common Unicode artifacts
    t = (
        t.replace("\u2018", "'")
         .replace("\u2019", "'")
         .replace("\u201c", '"')
         .replace("\u201d", '"')
    )
    t = t.replace("&lt;",   " ") \
     .replace("&gt;",   " ") \
     .replace("&nbsp;", " ") \
     .replace("&apos;", "'") \
     .replace("&quot;", '"')
    # HTML entities
    t = re.sub(r"&#\d+;", " ", t)
    # Latin-1 / Windows-1252 artifacts
    t = re.sub(r"[ÃÂâ][^\s]*", " ", t)
    t = re.sub(r"\\x[0-9a-fA-F]{2}", "", t)
    # Whitespace normalization
    t = re.sub(r"\s+", " ", t).strip()
    return t

### **Detect Slang and Abbrev**

In [11]:
def detect_slang(text: str) -> int:
    text_lower = str(text).lower()

    # (a) Single-token whole-word matching
    tokens = set(re.findall(r'\b\w+\b', text_lower))
    if tokens & SLANG_WORDS:
        return 1

    # (b) Multi-word phrase substring matching
    for phrase in SLANG_PHRASES:
        if phrase in text_lower:
            return 1

    # (c) Indonesian nge- verbal prefix (distinctly informal)
    #     Pola: nge + kata kerja >= 3 karakter
    if re.search(r'\bnge[a-z]{3,}\b', text_lower):
        return 1

    return 0


def detect_abbrev(text: str) -> int:
    text_lower = str(text).lower()
    tokens = set(re.findall(r'\b\w+\b', text_lower))

    # (a) Kamus abbrev (whole-word match)
    if tokens & ABBREV_WORDS:
        return 1

    # (b) Leet-speak: huruf + digit + huruf (4nj1ng, b4ngs4t, g2g)
    if re.search(r'\b(?:[a-z]+[0-9][a-z0-9]*|[0-9]+[a-z][a-z0-9]*)\b', text_lower):
        return 1

    # (c) Reduplikasi alay Indonesia: kata + angka 2 (jalan2, temen2)
    if re.search(r'[a-z]{2,}2(?:\b|$)', text_lower):
        return 1

    return 0

### **Multi-Dimensional Stratified Sampling**

In [12]:
def _multi_stratified_sample(df: pd.DataFrame, n_per_class: int, seed: int) -> pd.DataFrame:
    groups = []

    for lbl in sorted(df["label"].unique()):
        subset = df[df["label"] == lbl].copy()
        n = min(n_per_class, len(subset))
        if n < n_per_class:
            print(f"  [!] WARN: label={lbl} hanya {len(subset):,} baris tersedia, "
                  f"menggunakan semua.")

        sampled = None

        # ── Coba skmultilearn (Iterative Stratification) ──────────────
        try:
            from skmultilearn.model_selection import iterative_train_test_split

            X = np.arange(len(subset)).reshape(-1, 1)
            y = subset[["has_slang", "has_abbrev"]].values.astype(float)

            if n >= len(subset):
                sampled = subset
            else:
                test_ratio = n / len(subset)
                # iterative_train_test_split returns (X_train, y_train, X_test, y_test)
                _, _, X_sel, _ = iterative_train_test_split(X, y, test_size=test_ratio)
                selected_idx = set(X_sel.flatten())
                sampled = subset.iloc[sorted(selected_idx)]

                shortfall = n - len(sampled)
                if 0 < shortfall:
                    remaining = subset.iloc[
                        [i for i in range(len(subset)) if i not in selected_idx]
                    ]
                    extra = remaining.sample(
                        n=min(shortfall, len(remaining)),
                        random_state=seed
                    )
                    sampled = pd.concat([sampled, extra]).reset_index(drop=True)
                    print(f"  [top-up] +{len(extra)} baris untuk tutup shortfall label={lbl}")

        except Exception as e:
            print(f"  [!] skmultilearn unavailable ({e}); fallback ke composite stratum.")

        # ── Fallback: composite stratum via sklearn ─────────────────────
        if sampled is None:
            subset["_strat"] = (
                subset["has_slang"].astype(str) + "_" + subset["has_abbrev"].astype(str)
            )
            strat_col  = subset["_strat"]
            counts     = strat_col.value_counts()
            can_strat  = (counts >= 2).all() and n < len(subset)

            if can_strat:
                try:
                    from sklearn.model_selection import train_test_split as sk_split
                    _, sampled = sk_split(
                        subset, test_size=n, stratify=strat_col, random_state=seed
                    )
                except Exception:
                    sampled = subset.sample(n=n, random_state=seed)
            else:
                sampled = subset.sample(n=n, random_state=seed)

            sampled = sampled.drop(columns=["_strat"], errors="ignore")

        groups.append(sampled)

    result = pd.concat(groups).sample(frac=1, random_state=seed).reset_index(drop=True)
    return result

### **Full Cleaning**

In [13]:
def clean_text(text):
    if pd.isna(text):
        return ""
    t = str(text).strip().lower()

    # Encoding fix
    t = re.sub(r"\\[ntr]", " ", t)
    t = re.sub(r"(\\\s*)+", " ", t)
    t = t.replace("&amp;", " dan ")

    # Elongation: maks 2 karakter berulang (anjiiir → anjir)
    t = re.sub(r"([^.!?])\1{2,}", r"\1\1", t)

    # Sisa encoding noise
    t = re.sub(r"\bx[0-9]{2,3}\b", " ", t)
    t = re.sub(r"[\x00-\x1f\x7f-\x9f]", " ", t)

    # Twitter/X artifacts: RT
    t = re.sub(r"(?i)^rt\s+", "", t)
    t = re.sub(r"(?i)\brt\b", " ", t)
    t = re.sub(r"(?i)\bretweeted\b.*?\buser\b\)?\s*[:;]*", " ", t)

    t = re.sub(r"@\w+", " user ", t)

    # Hapus URL
    t = re.sub(r"http\S+|www\.\S+|t\.co/\S+", " ", t)

    # Hashtag: hapus #, pertahankan teks
    t = re.sub(r"#(\w+)", r"\1", t)

    # Hapus emoji & karakter di luar BMP
    t = re.sub(r"[\U00010000-\U0010ffff]", " ", t)

    # Hapus angka berdiri sendiri (bukan bagian leet-speak)
    # '4nj1ng': 4 diikuti huruf -> TIDAK dihapus
    # 'meet at 4': 4 diapit spasi -> DIHAPUS
    t = re.sub(r"(?<![a-z])\d+(?![a-z])", " ", t)

    # Tanda petik berlebih
    t = t.replace('"', " ")
    t = re.sub(r"(?<![a-zA-Z])'", " ", t)
    t = re.sub(r"'(?![a-zA-Z])", " ", t)

    # Tanda baca berulang
    t = re.sub(r"\?{2,}", " ?", t)
    t = re.sub(r"!{2,}", " !", t)
    t = re.sub(r"\.{4,}", "...", t)
    t = re.sub(r"[:;]{2,}", " ", t)

    # Deduplikasi 'user user' → 'user'
    t = re.sub(r"(?:\buser\b\s*){2,}", "user ", t)

    # Hapus karakter non-alfanumerik kecuali yang relevan
    t = re.sub(r"[^a-zA-Z0-9\s',\.?!]", " ", t)

    # Sisa apostrof gantung
    t = re.sub(r"(?<![a-zA-Z])'", " ", t)
    t = re.sub(r"'(?![a-zA-Z])", " ", t)

    # Normalize whitespace
    t = re.sub(r"\s+", " ", t).strip()
    return t

### **Pipeline**

In [14]:
def run_pipeline_indonesia(
    input_path: str,
    output_path: str,
    text_col: str    = "text",
    label_col: str   = "labels",
    source: str      = "tonneau_indonesian",
    n_per_class: int = 6000,
    seed: int        = 42
):
    LANGUAGE      = "id"
    BUFFER_RATIO  = 1.15       # 15% buffer mengkompensasi data loss saat cleaning
    DRIFT_THRESH  = 0.05       # threshold: pergeseran distribusi > 5%

    SEP_MAIN = "=" * 62
    SEP_SUB  = "─" * 55

    print(f"\n{SEP_MAIN}")
    print(f"  PIPELINE: BAHASA INDONESIA | target akhir: {n_per_class * 2:,} baris")
    print(f"{SEP_MAIN}")

    # ────────────────────────────────────────────────────────────────────
    # [1] LOAD RAW DATA
    # ────────────────────────────────────────────────────────────────────
    print(f"\n[1] LOAD RAW DATA")
    df = pd.read_csv(input_path)
    before_na = len(df)
    print(f"    File            : {input_path}")
    print(f"    Baris dimuat    : {before_na:>7,}")
    print(f"    Kolom           : {list(df.columns)}")

    df = df[[text_col, label_col]].copy()
    df = df.dropna(subset=[text_col, label_col])
    print(f"    Hapus missing   : {before_na - len(df)}")

    df = df.rename(columns={text_col: "text", label_col: "label"})
    df["label"] = pd.to_numeric(df["label"], errors="coerce")
    before_nb = len(df)
    df = df.dropna(subset=["label"])
    df["label"] = df["label"].astype(int)
    df = df[df["label"].isin([0, 1])].reset_index(drop=True)
    print(f"    Hapus non-binary: {before_nb - len(df)}")

    df["text"] = df["text"].astype(str).str.strip()
    before_empty = len(df)
    df = df[df["text"] != ""].reset_index(drop=True)
    print(f"    Hapus kosong    : {before_empty - len(df)}")
    print(f"    Distribusi label: {df['label'].value_counts().sort_index().to_dict()}")
    print(f"    Total valid     : {len(df):,} baris")
    n_after_load = len(df)

    # ────────────────────────────────────────────────────────────────────
    # [2] MINIMAL CLEANING (encoding fix + normalisasi whitespace)
    # ────────────────────────────────────────────────────────────────────
    print(f"\n[2] MINIMAL CLEANING (encoding fix + normalisasi whitespace)")
    tqdm.pandas(desc="    minimal_clean  ")
    df["_text_mc"] = df["text"].progress_apply(minimal_clean)
    print(f"    Dilakukan pada  : {len(df):,} baris")
    print(f"    Contoh output   : {repr(df['_text_mc'].iloc[0][:80])}")
    n_after_minimal = len(df)

    # ────────────────────────────────────────────────────────────────────
    # [3] FEATURE DETECTION (has_slang | has_abbrev)
    # ────────────────────────────────────────────────────────────────────
    print(f"\n[3] FEATURE DETECTION (has_slang | has_abbrev)")
    tqdm.pandas(desc="    detect_slang   ")
    df["has_slang"]  = df["_text_mc"].progress_apply(detect_slang)
    tqdm.pandas(desc="    detect_abbrev  ")
    df["has_abbrev"] = df["_text_mc"].progress_apply(detect_abbrev)

    n_sl = df["has_slang"].sum()
    n_ab = df["has_abbrev"].sum()
    n11  = ((df["has_slang"]==1) & (df["has_abbrev"]==1)).sum()
    n10  = ((df["has_slang"]==1) & (df["has_abbrev"]==0)).sum()
    n01  = ((df["has_slang"]==0) & (df["has_abbrev"]==1)).sum()
    n00  = ((df["has_slang"]==0) & (df["has_abbrev"]==0)).sum()
    print(f"    has_slang       : {n_sl:,} ({df['has_slang'].mean()*100:.1f}%)")
    print(f"    has_abbrev      : {n_ab:,} ({df['has_abbrev'].mean()*100:.1f}%)")
    print(f"    (slang, abbrev) : (1,1)={n11}  (1,0)={n10}  (0,1)={n01}  (0,0)={n00}")
    if df["has_slang"].mean() > 0.95:
        print("    [!] PERINGATAN: has_slang rate > 95% — review SLANG_WORDS (kemungkinan false positive).")
    n_after_feature = len(df)

    # ────────────────────────────────────────────────────────────────────
    # [4] DISTRIBUTION ANALYSIS (pre-sampling, dicatat untuk Bab 3)
    # ────────────────────────────────────────────────────────────────────
    print(f"\n[4] DISTRIBUTION ANALYSIS (pre-sampling — dicatat untuk Bab 3)")
    print(f"    {SEP_SUB}")
    for lbl_val, lbl_name in [(0, "NON-HATE"), (1, "HATE   ")]:
        sub  = df[df["label"] == lbl_val]
        n_s  = sub["has_slang"].sum()
        n_a  = sub["has_abbrev"].sum()
        n_11 = ((sub["has_slang"]==1) & (sub["has_abbrev"]==1)).sum()
        n_00 = ((sub["has_slang"]==0) & (sub["has_abbrev"]==0)).sum()
        pct  = max(len(sub), 1)
        print(f"    {lbl_name} (n={len(sub):6,}): "
              f"slang={n_s}({n_s/pct*100:.1f}%)  "
              f"abbrev={n_a}({n_a/pct*100:.1f}%)  "
              f"(1,1)={n_11}  (0,0)={n_00}")
    print(f"    {SEP_SUB}")
    pre_label_dist  = df["label"].value_counts(normalize=True).sort_index().to_dict()
    pre_slang_rate  = df["has_slang"].mean()
    pre_abbrev_rate = df["has_abbrev"].mean()
    print(f"    Proporsi label  : {pre_label_dist}")
    print(f"    Rate has_slang  : {pre_slang_rate:.4f}")
    print(f"    Rate has_abbrev : {pre_abbrev_rate:.4f}")

    # ────────────────────────────────────────────────────────────────────
    # [5] MULTI-DIMENSIONAL STRATIFIED SAMPLING
    #     Strata: label × has_slang × has_abbrev
    #     Metode: Iterative Stratification (Sechidis et al., 2011)
    #     Buffer: ambil n_buffer/kelas > target → trim ke tepat n_per_class setelah cleaning
    # ────────────────────────────────────────────────────────────────────
    n_buffer = int(round(n_per_class * BUFFER_RATIO))
    print(f"\n[5] MULTI-DIMENSIONAL STRATIFIED SAMPLING")
    print(f"    Strata  : label × has_slang × has_abbrev")
    print(f"    Metode  : Iterative Stratification (Sechidis et al., 2011)")
    print(f"    Target  : {n_per_class:,}/kelas = {n_per_class*2:,} total")
    print(f"    Buffer  : {n_buffer:,}/kelas (BUFFER_RATIO={BUFFER_RATIO}) → akan di-trim setelah cleaning")
    before_samp = len(df)
    df = _multi_stratified_sample(df, n_buffer, seed)
    print(f"    {before_samp:,} → {len(df):,} baris (sampling dengan buffer)")
    print(f"    Distribusi label: {df['label'].value_counts().sort_index().to_dict()}")
    n11 = ((df["has_slang"]==1) & (df["has_abbrev"]==1)).sum()
    n10 = ((df["has_slang"]==1) & (df["has_abbrev"]==0)).sum()
    n01 = ((df["has_slang"]==0) & (df["has_abbrev"]==1)).sum()
    n00 = ((df["has_slang"]==0) & (df["has_abbrev"]==0)).sum()
    print(f"    (slang, abbrev) : (1,1)={n11}  (1,0)={n10}  (0,1)={n01}  (0,0)={n00}")
    n_after_sampling = len(df)

    # ────────────────────────────────────────────────────────────────────
    # [6] FULL DATA CLEANING
    # ────────────────────────────────────────────────────────────────────
    print(f"\n[6] FULL DATA CLEANING")

    # [6a] Dedup teks mentah
    before_dup = len(df)
    df = df.drop_duplicates(subset=["text"], keep="first").reset_index(drop=True)
    n_after_dedup_raw = len(df)
    print(f"    [6a] Dedup teks mentah        : {before_dup:,} → {n_after_dedup_raw:,} "
          f"(hapus {before_dup - n_after_dedup_raw})")

    # [6b] clean_text (pada _text_mc)
    print("    [6b] Cleaning teks...")
    tqdm.pandas(desc="         clean_text     ")
    df["_text_clean"] = df["_text_mc"].progress_apply(clean_text)
    n_after_clean = len(df)
    print(f"         Selesai               : {n_after_clean:,} baris")

    # [6c] Post-cleaning filter (teks kosong + media-only + dedup cleaned)
    media_only_pat = r"(?i)\bmedia\s*only\b.*\bno\s*text\b"
    before_postcl = len(df)
    df = df[df["_text_clean"].str.strip().ne("")]
    df = df[~df["_text_clean"].str.contains(media_only_pat, regex=True, na=False)]
    df = df[df["_text_clean"].str.contains(r"[a-zA-Z0-9]", regex=True, na=False)]   # ← pengganti semantik
    df = df.drop_duplicates(subset=["_text_clean"], keep="first").reset_index(drop=True)
    n_after_postcl = len(df)
    print(f"    [6c] Filter kosong + dedup   : {before_postcl:,} → {n_after_postcl:,} "
          f"(hapus {before_postcl - n_after_postcl})")

    # [6d] Filter panjang token — outlier removal (< 3 atau > 512 token)
    df["_token_len"] = df["_text_clean"].apply(lambda t: len(t.split()))
    before_len = len(df)
    df = df[(df["_token_len"] >= 3) & (df["_token_len"] <= 512)]
    df = df.drop(columns=["_token_len"]).reset_index(drop=True)
    n_after_lenfilter = len(df)
    print(f"    [6d] Filter panjang (3–512)  : {before_len:,} → {n_after_lenfilter:,} "
          f"(hapus {before_len - n_after_lenfilter})")

    # ────────────────────────────────────────────────────────────────────
    # [7] POST-CLEANING FILTER + VERIFIKASI DISTRIBUSI
    # ────────────────────────────────────────────────────────────────────
    print(f"\n[7] POST-CLEANING FILTER + VERIFIKASI DISTRIBUSI")

    # Shuffle sebelum trim agar seleksi akhir bersifat random
    df = df.sample(frac=1, random_state=seed).reset_index(drop=True)

    # Trim ke tepat n_per_class per label → total persis n_per_class * 2
    trimmed = []
    for lbl in [0, 1]:
        group = df[df["label"] == lbl]
        if len(group) >= n_per_class:
            trimmed.append(group.head(n_per_class))
        else:
            print(f"    [!] WARNING: label={lbl} hanya {len(group):,} baris "
                  f"(target: {n_per_class:,}) — menggunakan semua yang tersedia.")
            trimmed.append(group)
    df = pd.concat(trimmed).sample(frac=1, random_state=seed).reset_index(drop=True)
    n_final = len(df)
    print(f"    Trim ke target  : {n_after_lenfilter:,} → {n_final:,} baris")
    print(f"    Distribusi label: {df['label'].value_counts().sort_index().to_dict()}")

    # Verifikasi drift distribusi pre-sampling vs post-cleaning
    post_label_dist  = df["label"].value_counts(normalize=True).sort_index().to_dict()
    post_slang_rate  = df["has_slang"].mean()
    post_abbrev_rate = df["has_abbrev"].mean()
    print(f"\n    Verifikasi drift distribusi (threshold Δ > {DRIFT_THRESH:.0%})")
    print(f"    {SEP_SUB}")
    drifted = False
    for lbl_val in [0, 1]:
        pre_v  = pre_label_dist.get(lbl_val, 0)
        post_v = post_label_dist.get(lbl_val, 0)
        delta  = abs(post_v - pre_v)
        flag   = " [!] DRIFT" if delta > DRIFT_THRESH else " ✓"
        print(f"    Label={lbl_val}: pre={pre_v:.4f} → post={post_v:.4f}  Δ={delta:.4f}{flag}")
        if delta > DRIFT_THRESH:
            drifted = True
    s_delta = abs(post_slang_rate  - pre_slang_rate)
    a_delta = abs(post_abbrev_rate - pre_abbrev_rate)
    flag_s  = " [!] DRIFT" if s_delta > DRIFT_THRESH else " ✓"
    flag_a  = " [!] DRIFT" if a_delta > DRIFT_THRESH else " ✓"
    print(f"    has_slang : pre={pre_slang_rate:.4f} → post={post_slang_rate:.4f}  Δ={s_delta:.4f}{flag_s}")
    print(f"    has_abbrev: pre={pre_abbrev_rate:.4f} → post={post_abbrev_rate:.4f}  Δ={a_delta:.4f}{flag_a}")
    if s_delta > DRIFT_THRESH: drifted = True
    if a_delta > DRIFT_THRESH: drifted = True
    print(f"    {SEP_SUB}")
    if drifted:
        print("    [!] Distribusi bergeser signifikan. Pertimbangkan re-sampling atau")
        print("        dokumentasikan sebagai limitasi di Bab 3.")
    else:
        print("    ✓  Tidak ada drift signifikan. Distribusi terjaga dengan baik.")

    # Tabel transisi data (wajib dilaporkan di Bab 3)
    print(f"\n    TABEL TRANSISI DATA (Bab 3 — Tabel transisi_cleaning)")
    print(f"    {SEP_SUB}")
    print(f"    [1] Data mentah (raw CSV)                       : {before_na:>7,}")
    print(f"    [2] Setelah validasi (NaN / kosong / non-binary): {n_after_load:>7,}")
    print(f"    [3] Setelah minimal cleaning                    : {n_after_minimal:>7,}")
    print(f"    [4] Setelah feature detection                   : {n_after_feature:>7,}")
    print(f"    [5] Setelah stratified sampling (buffer ×{BUFFER_RATIO})   : {n_after_sampling:>7,}")
    print(f"    [6a] Setelah dedup teks mentah                  : {n_after_dedup_raw:>7,}")
    print(f"    [6b] Setelah full cleaning                       : {n_after_clean:>7,}")
    print(f"    [6c] Setelah post-cleaning filter               : {n_after_postcl:>7,}")
    print(f"    [6d] Setelah filter panjang token (3–512)       : {n_after_lenfilter:>7,}")
    print(f"    [7]  Setelah trim ke target                     : {n_final:>7,}")
    print(f"    {SEP_SUB}")
    print(f"    FINAL DATASET                                   : {n_final:>7,}")

    # ────────────────────────────────────────────────────────────────────
    # [8] SAVE
    # ────────────────────────────────────────────────────────────────────
    print(f"\n[8] SAVE")

    # Assign ID sequential mulai dari 1 (bukan UUID)
    df["id"]       = range(1, len(df) + 1)
    df["language"] = LANGUAGE
    df["source"]   = source

    # [8b] Final CSV — tepat 7 kolom sesuai skema Bab 3
    df_final = pd.DataFrame({
        "id"        : df["id"],
        "text"      : df["_text_clean"],  # bloom-cleaned, lowercased, siap BLOOM
        "label"     : df["label"],
        "language"  : df["language"],
        "has_slang" : df["has_slang"],
        "has_abbrev": df["has_abbrev"],
        "source"    : df["source"],
    })
    assert list(df_final.columns) == [
        "id", "text", "label", "language", "has_slang", "has_abbrev", "source"
    ], f"[ERROR] Kolom tidak sesuai: {list(df_final.columns)}"

    df_final.to_csv(output_path, index=False)
    print(f"    [8b] Final CSV   : {output_path}")
    print(f"         Kolom       : {list(df_final.columns)}")

    # Ringkasan akhir
    n11 = ((df_final["has_slang"]==1) & (df_final["has_abbrev"]==1)).sum()
    n10 = ((df_final["has_slang"]==1) & (df_final["has_abbrev"]==0)).sum()
    n01 = ((df_final["has_slang"]==0) & (df_final["has_abbrev"]==1)).sum()
    n00 = ((df_final["has_slang"]==0) & (df_final["has_abbrev"]==0)).sum()
    print(f"\n{SEP_MAIN}")
    print(f"  RINGKASAN AKHIR — BAHASA INDONESIA")
    print(f"{SEP_MAIN}")
    print(f"  Total baris      : {len(df_final):,}")
    print(f"  Distribusi label : {df_final['label'].value_counts().sort_index().to_dict()}")
    print(f"  has_slang        : {df_final['has_slang'].sum():,} ({df_final['has_slang'].mean()*100:.1f}%)")
    print(f"  has_abbrev       : {df_final['has_abbrev'].sum():,} ({df_final['has_abbrev'].mean()*100:.1f}%)")
    print(f"  (slang, abbrev)  : (1,1)={n11}  (1,0)={n10}  (0,1)={n01}  (0,0)={n00}")
    print(f"  ID range         : {df_final['id'].iloc[0]} – {df_final['id'].iloc[-1]}")
    print(f"  Kolom output     : {list(df_final.columns)}")
    print(f"  Contoh 5 baris pertama:")
    print(df_final[["id", "text", "label", "has_slang", "has_abbrev"]].head().to_string())
    print(f"{SEP_MAIN}")

    return df_final

### **Run Code**

In [15]:
df_id = run_pipeline_indonesia(
    input_path  = '/content/drive/MyDrive/Colab Datasets/in_hf.csv',
    output_path = '/content/drive/MyDrive/Colab Datasets/dataset_indonesia_final.csv',
    text_col    = 'text',
    label_col   = 'labels',
    source      = 'tonneau_indonesian',
    n_per_class = 6000,   # 6.000 hate + 6.000 non-hate = 12.000 total ID
    seed        = 42
)


  PIPELINE: BAHASA INDONESIA | target akhir: 12,000 baris

[1] LOAD RAW DATA
    File            : /content/drive/MyDrive/Colab Datasets/in_hf.csv
    Baris dimuat    :  14,306
    Kolom           : ['text', 'labels', 'source', 'dataset', 'nb_annotators']
    Hapus missing   : 0
    Hapus non-binary: 0
    Hapus kosong    : 0
    Distribusi label: {0: 8256, 1: 6050}
    Total valid     : 14,306 baris

[2] MINIMAL CLEANING (encoding fix + normalisasi whitespace)


    minimal_clean  : 100%|██████████| 14306/14306 [00:00<00:00, 31341.22it/s]


    Dilakukan pada  : 14,306 baris
    Contoh output   : '@USER.wood17 knp lo gak berani bersumpah dan bertaruh? Krnlo pecundang. Lo mau l'

[3] FEATURE DETECTION (has_slang | has_abbrev)


    detect_abbrev  : 100%|██████████| 14306/14306 [00:00<00:00, 25323.42it/s]


    has_slang       : 5,041 (35.2%)
    has_abbrev      : 7,381 (51.6%)
    (slang, abbrev) : (1,1)=3149  (1,0)=1892  (0,1)=4232  (0,0)=5033

[4] DISTRIBUTION ANALYSIS (pre-sampling — dicatat untuk Bab 3)
    ───────────────────────────────────────────────────────
    NON-HATE (n= 8,256): slang=2690(32.6%)  abbrev=3985(48.3%)  (1,1)=1660  (0,0)=3241
    HATE    (n= 6,050): slang=2351(38.9%)  abbrev=3396(56.1%)  (1,1)=1489  (0,0)=1792
    ───────────────────────────────────────────────────────
    Proporsi label  : {0: 0.5771005172654831, 1: 0.422899482734517}
    Rate has_slang  : 0.3524
    Rate has_abbrev : 0.5159

[5] MULTI-DIMENSIONAL STRATIFIED SAMPLING
    Strata  : label × has_slang × has_abbrev
    Metode  : Iterative Stratification (Sechidis et al., 2011)
    Target  : 6,000/kelas = 12,000 total
    Buffer  : 6,900/kelas (BUFFER_RATIO=1.15) → akan di-trim setelah cleaning
  [!] WARN: label=1 hanya 6,050 baris tersedia, menggunakan semua.
    14,306 → 12,950 baris (sampling den

         clean_text     : 100%|██████████| 12830/12830 [00:02<00:00, 5127.51it/s]


         Selesai               : 12,830 baris
    [6c] Filter kosong + dedup   : 12,830 → 12,689 (hapus 141)
    [6d] Filter panjang (3–512)  : 12,689 → 12,585 (hapus 104)

[7] POST-CLEANING FILTER + VERIFIKASI DISTRIBUSI
    [!] WARNING: label=1 hanya 5,794 baris (target: 6,000) — menggunakan semua yang tersedia.
    Trim ke target  : 12,585 → 11,794 baris
    Distribusi label: {0: 6000, 1: 5794}

    Verifikasi drift distribusi (threshold Δ > 5%)
    ───────────────────────────────────────────────────────
    Label=0: pre=0.5771 → post=0.5087  Δ=0.0684 [!] DRIFT
    Label=1: pre=0.4229 → post=0.4913  Δ=0.0684 [!] DRIFT
    has_slang : pre=0.3524 → post=0.3581  Δ=0.0058 ✓
    has_abbrev: pre=0.5159 → post=0.5237  Δ=0.0077 ✓
    ───────────────────────────────────────────────────────
    [!] Distribusi bergeser signifikan. Pertimbangkan re-sampling atau
        dokumentasikan sebagai limitasi di Bab 3.

    TABEL TRANSISI DATA (Bab 3 — Tabel transisi_cleaning)
    ─────────────────────